# Customer Request & Ground Truth Generation

## Objective

Generate synthetic customer requests in natural language and
their corresponding structured reference requirements.

The dataset will be used to train and evaluate an NLP model
that extracts:

- Characteristic
- Value
- Operator (EQUAL, MIN, MAX)
- Requirement type (MUST, PREFERRED, INDIFFERENT)

## Data Sources

- CHARACTERISTIC_DICTIONARY
- CHARACTERISTIC_VALUES
- CONFIGURATION_RULES
- PRODUCT_MASTER

## Approach

1. Generate structured customer requirements.
2. Convert requirements into natural-language requests.
3. Validate consistency between requests and ground truth.
4. Create training, validation, and test datasets.

In [4]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "oven_configurator_master_v1.0.xlsx"

print("Master file exists:", MASTER_PATH.exists())

# Load master data
dictionary = pd.read_excel(MASTER_PATH, sheet_name="CHARACTERISTIC_DICTIONARY")
values = pd.read_excel(MASTER_PATH, sheet_name="CHARACTERISTIC_VALUES")
rules = pd.read_excel(MASTER_PATH, sheet_name="CONFIGURATION_RULES")

# Inspect dataset structure
for name, df in [
    ("CHARACTERISTIC_DICTIONARY", dictionary),
    ("CHARACTERISTIC_VALUES", values),
    ("CONFIGURATION_RULES", rules),
]:
    print(f"\n{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    display(df.head(3))

Master file exists: True

CHARACTERISTIC_DICTIONARY: (25, 6)
Columns: ['-', 'NUMERIC / CATEGORICAL / BOOLEAN / ORDINAL', '-.1', 'EXACT / DISTANCE / ORDINAL', 'cm, L, adet vb.', '-.2']


,-,NUMERIC / CATEGORICAL / BOOLEAN / ORDINAL,-.1,EXACT / DISTANCE / ORDINAL,"cm, L, adet vb.",-.2
0,CHARACTERISTIC,DATA_TYPE,ALLOWED_VALUES / RANGE,MATCH_METHOD,UNIT,DESCRIPTION
1,WIDTH_CM,NUMERIC,"45, 60, 70, 90",EXACT,cm,Product installation width
2,PACKAGING_TYPE,CATEGORICAL,"SHRINK, BOX",EXACT,-,Design of product packaging



CHARACTERISTIC_VALUES: (90, 3)
Columns: ['CHARACTERISTIC', 'VALUE', 'DEFINITION']


,CHARACTERISTIC,VALUE,DEFINITION
0,WIDTH_CM,V001,45
1,WIDTH_CM,V002,60
2,WIDTH_CM,V003,70



CONFIGURATION_RULES: (170, 6)
Columns: ['RuleId', 'Type', 'Operator', 'CHARACTERISTIC', 'VALUE', 'DEFINITION']


,RuleId,Type,Operator,CHARACTERISTIC,VALUE,DEFINITION
0,1,Auto Select,Action,TURNSPIT,V001,YES
1,1,AND,Rule,OVEN_FUNCTIONS,V004,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
2,1,AND,Rule,OVEN_FUNCTIONS,V007,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...


In [5]:
# Read characteristic dictionary with the correct header row
dictionary = pd.read_excel(
    MASTER_PATH,
    sheet_name="CHARACTERISTIC_DICTIONARY",
    header=1
)

print("Dictionary shape:", dictionary.shape)
print("Columns:", dictionary.columns.tolist())

display(dictionary)

Dictionary shape: (24, 6)
Columns: ['CHARACTERISTIC', 'DATA_TYPE', 'ALLOWED_VALUES / RANGE', 'MATCH_METHOD', 'UNIT', 'DESCRIPTION']


,CHARACTERISTIC,DATA_TYPE,ALLOWED_VALUES / RANGE,MATCH_METHOD,UNIT,DESCRIPTION
0,WIDTH_CM,NUMERIC,"45, 60, 70, 90",EXACT,cm,Product installation width
1,PACKAGING_TYPE,CATEGORICAL,"SHRINK, BOX",EXACT,-,Design of product packaging
2,PANEL_TYPE,CATEGORICAL,"GLASS, METAL",EXACT,-,Control panel material
3,PANEL_COLOR,CATEGORICAL,"BLACK, WHITE, SILVER, STAINLESS STEEL, DARK SI...",EXACT,-,Control panel color
4,POWER_CABLE,CATEGORICAL,"W/O CABLE, W/O PLUG, EU TYPE PLUG, US TYPE PLU...",EXACT,-,Power cable availability of product based on p...
5,WIFI,BOOLEAN,"YES, NO",EXACT,-,Wireless connectivity
6,TURNSPIT,BOOLEAN,"YES, NO",EXACT,-,A motorized accessory used for slow-roasting l...
7,OVEN_TYPE,CATEGORICAL,"CONVECTION, STEAM, COMBINATION",EXACT,-,Categorization of built in oven by how it gene...
8,NUMBER_OF_FUNCTIONS,NUMERIC,"3, 5, 8, 10, 11",DISTANCE,quantity,Number of cooking modes
9,OVEN_FUNCTIONS,CATEGORICAL,01-12 VALUES ARE AVAILABLE,EXACT,-,Preset cooking function set ID


In [9]:
display(
    dictionary[
        ["CHARACTERISTIC", "DATA_TYPE", "MATCH_METHOD"]
    ]
)

,CHARACTERISTIC,DATA_TYPE,MATCH_METHOD
0,WIDTH_CM,NUMERIC,EXACT
1,PACKAGING_TYPE,CATEGORICAL,EXACT
2,PANEL_TYPE,CATEGORICAL,EXACT
3,PANEL_COLOR,CATEGORICAL,EXACT
4,POWER_CABLE,CATEGORICAL,EXACT
5,WIFI,BOOLEAN,EXACT
6,TURNSPIT,BOOLEAN,EXACT
7,OVEN_TYPE,CATEGORICAL,EXACT
8,NUMBER_OF_FUNCTIONS,NUMERIC,DISTANCE
9,OVEN_FUNCTIONS,CATEGORICAL,EXACT


In [10]:
# Step 5 - First customer request and reference requirements

request_id = "R0001"

customer_request = pd.DataFrame([
    {
        "REQUEST_ID": request_id,
        "REQUEST_TEXT": (
            "60 cm genişliğinde, siyah panelli bir fırın istiyorum. "
            "Wi-Fi özelliği olursa iyi olur."
        )
    }
])

requirements = [
    ("WIDTH_CM", "60", "EQUAL", "MUST"),
    ("PANEL_COLOR", "BLACK", "EQUAL", "MUST"),
    ("WIFI", "YES", "EQUAL", "PREFERRED")
]

ground_truth = pd.DataFrame(
    [
        {
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": value,
            "OPERATOR": operator,
            "REQUIREMENT_TYPE": requirement_type
        }
        for characteristic, value, operator, requirement_type
        in requirements
    ]
)

display(customer_request)
display(ground_truth)

,REQUEST_ID,REQUEST_TEXT
0,R0001,"60 cm genişliğinde, siyah panelli bir fırın is..."


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,R0001,WIDTH_CM,60,EQUAL,MUST
1,R0001,PANEL_COLOR,BLACK,EQUAL,MUST
2,R0001,WIFI,YES,EQUAL,PREFERRED


In [11]:
import random

# Reproducible random selection
rng = random.Random(42)

# Characteristics for our first pilot
pilot_characteristics = [
    "WIDTH_CM",
    "PANEL_COLOR",
    "CLEANING",
    "WIFI"
]

# Requirement type probabilities
requirement_types = ["MUST", "PREFERRED"]
requirement_weights = [0.75, 0.25]

generated_requirements = []

for characteristic in pilot_characteristics:

    # Find allowed values for this characteristic
    available_values = values[
        values["CHARACTERISTIC"] == characteristic
    ]

    # Randomly select one valid value
    selected = available_values.sample(
        n=1,
        random_state=rng.randint(0, 100000)
    ).iloc[0]

    requirement_type = rng.choices(
        requirement_types,
        weights=requirement_weights,
        k=1
    )[0]

    generated_requirements.append({
        "REQUEST_ID": "R0002",
        "CHARACTERISTIC": characteristic,
        "VALUE": selected["VALUE"],
        "VALUE_DEFINITION": selected["DEFINITION"],
        "OPERATOR": "EQUAL",
        "REQUIREMENT_TYPE": requirement_type
    })

generated_ground_truth = pd.DataFrame(generated_requirements)

display(generated_ground_truth)

,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0002,WIDTH_CM,V001,45,EQUAL,MUST
1,R0002,PANEL_COLOR,V001,BLACK,EQUAL,MUST
2,R0002,CLEANING,V002,CATALYTIC,EQUAL,MUST
3,R0002,WIFI,V001,YES,EQUAL,MUST


In [14]:
# Step 7 - Convert structured requirements into English customer language

def requirement_to_text(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"])
    requirement_type = row["REQUIREMENT_TYPE"]

    if characteristic == "WIDTH_CM":
        phrase = f"a {value} cm wide oven"

    elif characteristic == "PANEL_COLOR":
        phrase = f"a {value.lower()} control panel"

    elif characteristic == "CLEANING":
        phrase = f"{value.lower()} cleaning"

    elif characteristic == "WIFI":
        phrase = (
            "Wi-Fi connectivity"
            if value.upper() == "YES"
            else "no Wi-Fi connectivity"
        )

    else:
        raise ValueError(f"Unsupported characteristic: {characteristic}")

    if requirement_type == "MUST":
        return f"I need {phrase}."

    elif requirement_type == "PREFERRED":
        return f"I would prefer {phrase}."

    else:
        raise ValueError(f"Unsupported requirement type: {requirement_type}")


sentences = [
    requirement_to_text(row)
    for _, row in generated_ground_truth.iterrows()
]

request_text = " ".join(sentences)

generated_customer_request = pd.DataFrame([
    {
        "REQUEST_ID": "R0002",
        "REQUEST_TEXT": request_text
    }
])

print(request_text)
display(generated_customer_request)

I need a 45 cm wide oven. I need a black control panel. I need catalytic cleaning. I need Wi-Fi connectivity.


,REQUEST_ID,REQUEST_TEXT
0,R0002,I need a 45 cm wide oven. I need a black contr...


In [15]:
# Step 8 - Test mixed requirement types

mixed_ground_truth = generated_ground_truth.copy()

mixed_ground_truth.loc[
    mixed_ground_truth["CHARACTERISTIC"] == "WIFI",
    "REQUIREMENT_TYPE"
] = "PREFERRED"

mixed_sentences = [
    requirement_to_text(row)
    for _, row in mixed_ground_truth.iterrows()
]

mixed_request_text = " ".join(mixed_sentences)

print(mixed_request_text)

display(
    mixed_ground_truth[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "OPERATOR",
            "REQUIREMENT_TYPE"
        ]
    ]
)

I need a 45 cm wide oven. I need a black control panel. I need catalytic cleaning. I would prefer Wi-Fi connectivity.


,CHARACTERISTIC,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,WIDTH_CM,45,EQUAL,MUST
1,PANEL_COLOR,BLACK,EQUAL,MUST
2,CLEANING,CATALYTIC,EQUAL,MUST
3,WIFI,YES,EQUAL,PREFERRED


In [16]:
# Step 9 - Compose a more natural English customer request

must_phrases = []
preferred_phrases = []

for _, row in mixed_ground_truth.iterrows():
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"])

    if characteristic == "WIDTH_CM":
        phrase = f"{value} cm wide"
    elif characteristic == "PANEL_COLOR":
        phrase = f"a {value.lower()} control panel"
    elif characteristic == "CLEANING":
        phrase = f"{value.lower()} cleaning"
    elif characteristic == "WIFI":
        phrase = (
            "Wi-Fi connectivity"
            if value.upper() == "YES"
            else "no Wi-Fi connectivity"
        )
    else:
        raise ValueError(f"Unsupported characteristic: {characteristic}")

    if row["REQUIREMENT_TYPE"] == "MUST":
        must_phrases.append(phrase)
    elif row["REQUIREMENT_TYPE"] == "PREFERRED":
        preferred_phrases.append(phrase)

request_parts = []

if must_phrases:
    request_parts.append(
        "I'm looking for an oven that must be "
        + ", ".join(must_phrases)
        + "."
    )

if preferred_phrases:
    request_parts.append(
        "Ideally, it should also have "
        + ", ".join(preferred_phrases)
        + "."
    )

natural_request_text = " ".join(request_parts)

print(natural_request_text)

I'm looking for an oven that must be 45 cm wide, a black control panel, catalytic cleaning. Ideally, it should also have Wi-Fi connectivity.


In [17]:
# Step 10 - Generate grammatically correct requirement sentences

def compose_request(ground_truth):
    must_sentences = []
    preferred_sentences = []

    for _, row in ground_truth.iterrows():
        characteristic = row["CHARACTERISTIC"]
        value = str(row["VALUE_DEFINITION"])
        requirement_type = row["REQUIREMENT_TYPE"]

        if characteristic == "WIDTH_CM":
            phrase = f"be {value} cm wide"
        elif characteristic == "PANEL_COLOR":
            phrase = f"have a {value.lower()} control panel"
        elif characteristic == "CLEANING":
            phrase = f"have {value.lower()} cleaning"
        elif characteristic == "WIFI":
            phrase = (
                "have Wi-Fi connectivity"
                if value.upper() == "YES"
                else "have no Wi-Fi connectivity"
            )
        else:
            raise ValueError(f"Unsupported characteristic: {characteristic}")

        if requirement_type == "MUST":
            must_sentences.append(f"It must {phrase}.")
        elif requirement_type == "PREFERRED":
            preferred_sentences.append(f"I would prefer it to {phrase}.")
        else:
            raise ValueError(f"Unsupported requirement type: {requirement_type}")

    return "I'm looking for an oven. " + " ".join(
        must_sentences + preferred_sentences
    )


natural_request_v2 = compose_request(mixed_ground_truth)

print(natural_request_v2)

I'm looking for an oven. It must be 45 cm wide. It must have a black control panel. It must have catalytic cleaning. I would prefer it to have Wi-Fi connectivity.


In [18]:
# Step 11 - Prepare characteristic generation profile

characteristic_profile = dictionary[
    ["CHARACTERISTIC", "DATA_TYPE", "MATCH_METHOD"]
].copy()

characteristic_profile["REQUEST_WEIGHT"] = 1.0
characteristic_profile["ENABLED"] = True

display(characteristic_profile)


,CHARACTERISTIC,DATA_TYPE,MATCH_METHOD,REQUEST_WEIGHT,ENABLED
0,WIDTH_CM,NUMERIC,EXACT,1.0,True
1,PACKAGING_TYPE,CATEGORICAL,EXACT,1.0,True
2,PANEL_TYPE,CATEGORICAL,EXACT,1.0,True
3,PANEL_COLOR,CATEGORICAL,EXACT,1.0,True
4,POWER_CABLE,CATEGORICAL,EXACT,1.0,True
5,WIFI,BOOLEAN,EXACT,1.0,True
6,TURNSPIT,BOOLEAN,EXACT,1.0,True
7,OVEN_TYPE,CATEGORICAL,EXACT,1.0,True
8,NUMBER_OF_FUNCTIONS,NUMERIC,DISTANCE,1.0,True
9,OVEN_FUNCTIONS,CATEGORICAL,EXACT,1.0,True


In [19]:
# Step 12 - Define customer request generation weights

high_priority = [
    "WIDTH_CM",
    "PANEL_COLOR",
    "CLEANING",
    "OVEN_FUNCTIONS",
    "NUMBER_OF_FUNCTIONS",
    "OVEN_CONTROL",
    "OVEN_TYPE"
]

low_priority = [
    "PACKAGING_TYPE",
    "POWER_CABLE"
]

characteristic_profile["REQUEST_WEIGHT"] = 1.0

characteristic_profile.loc[
    characteristic_profile["CHARACTERISTIC"].isin(high_priority),
    "REQUEST_WEIGHT"
] = 3.0

characteristic_profile.loc[
    characteristic_profile["CHARACTERISTIC"].isin(low_priority),
    "REQUEST_WEIGHT"
] = 0.3

display(
    characteristic_profile[
        ["CHARACTERISTIC", "REQUEST_WEIGHT", "ENABLED"]
    ]
)

,CHARACTERISTIC,REQUEST_WEIGHT,ENABLED
0,WIDTH_CM,3.0,True
1,PACKAGING_TYPE,0.3,True
2,PANEL_TYPE,1.0,True
3,PANEL_COLOR,3.0,True
4,POWER_CABLE,0.3,True
5,WIFI,1.0,True
6,TURNSPIT,1.0,True
7,OVEN_TYPE,3.0,True
8,NUMBER_OF_FUNCTIONS,3.0,True
9,OVEN_FUNCTIONS,3.0,True


In [20]:
# Step 13 - Select characteristics for five customer requests

import random

rng = random.Random(42)

active_profile = characteristic_profile[
    characteristic_profile["ENABLED"]
].copy()

characteristics = active_profile["CHARACTERISTIC"].tolist()
weights = active_profile["REQUEST_WEIGHT"].tolist()


def select_characteristics(rng, min_count=2, max_count=6):
    """Select unique characteristics using weighted sampling."""

    count = rng.randint(min_count, max_count)

    available = list(zip(characteristics, weights))
    selected = []

    for _ in range(min(count, len(available))):
        chosen = rng.choices(
            available,
            weights=[item[1] for item in available],
            k=1
        )[0]

        selected.append(chosen[0])
        available.remove(chosen)

    return selected


request_profiles = []

for i in range(1, 6):
    request_id = f"R{i:04d}"

    selected = select_characteristics(rng)

    request_profiles.append({
        "REQUEST_ID": request_id,
        "N_CHARACTERISTICS": len(selected),
        "SELECTED_CHARACTERISTICS": selected
    })

request_profiles_df = pd.DataFrame(request_profiles)

display(request_profiles_df)

,REQUEST_ID,N_CHARACTERISTICS,SELECTED_CHARACTERISTICS
0,R0001,2,"[WIDTH_CM, OVEN_TYPE]"
1,R0002,3,"[PANEL_COLOR, PANEL_TYPE, ENERGY_CLASS]"
2,R0003,6,"[PACKAGING_TYPE, OVEN_FUNCTIONS, WIDTH_CM, OVE..."
3,R0004,3,"[HANDLE_TYPE, KNOB_COLOR, NUMBER_OF_FUNCTIONS]"
4,R0005,5,"[ROASTING_TRAY, TELESCOPIC_RAILS, WIDTH_CM, OV..."


In [ ]:
# Step 14 - Generate ground truth from valid product configurations (Product-Anchored Generation)

product_master = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "product_master.csv",
    dtype=str
)

value_lookup = values.copy()

# Normalize lookup keys
value_lookup["CHARACTERISTIC"] = (
    value_lookup["CHARACTERISTIC"].astype(str).str.strip()
)

value_lookup["VALUE"] = (
    value_lookup["VALUE"].astype(str).str.strip()
)

rng = random.Random(123)
ground_truth_rows = []

for _, request in request_profiles_df.iterrows():

    request_id = request["REQUEST_ID"]

    # Select a valid reference product
    product = product_master.sample(
        n=1,
        random_state=rng.randint(0, 100000)
    ).iloc[0]

    for characteristic in request["SELECTED_CHARACTERISTICS"]:

        # Product master contains VALUE codes (e.g. V001)
        product_value_code = str(product[characteristic]).strip()

        # Match by characteristic AND value code
        matching_values = value_lookup[
            (value_lookup["CHARACTERISTIC"] == characteristic)
            & (value_lookup["VALUE"] == product_value_code)
        ]

        if len(matching_values) != 1:
            raise ValueError(
                f"Expected one matching value: "
                f"{characteristic} = {product_value_code}, "
                f"found {len(matching_values)}"
            )

        selected_value = matching_values.iloc[0]

        requirement_type = rng.choices(
            ["MUST", "PREFERRED"],
            weights=[0.75, 0.25],
            k=1
        )[0]

        ground_truth_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": product_value_code,
            "VALUE_DEFINITION": selected_value["DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type
        })

pilot_ground_truth = pd.DataFrame(ground_truth_rows)

display(pilot_ground_truth)

,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0001,WIDTH_CM,V001,45,EQUAL,MUST
1,R0001,OVEN_TYPE,V001,CONVECTION,EQUAL,PREFERRED
2,R0002,PANEL_COLOR,V001,BLACK,EQUAL,MUST
3,R0002,PANEL_TYPE,V001,GLASS,EQUAL,PREFERRED
4,R0002,ENERGY_CLASS,V001,A,EQUAL,MUST
5,R0003,PACKAGING_TYPE,V002,BOX,EQUAL,MUST
6,R0003,OVEN_FUNCTIONS,V002,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...,EQUAL,MUST
7,R0003,WIDTH_CM,V002,60,EQUAL,MUST
8,R0003,OVEN_TYPE,V001,CONVECTION,EQUAL,MUST
9,R0003,KNOB_STRUCTURE,V001,ROTARY,EQUAL,MUST


In [ ]:
# Step 15 - Generate customer requirements independently of products

import random

rng = random.Random(456) #456 means that the random selection will be reproducible. You can change this number to get different random selections.

independent_rows = []

for i, request in request_profiles_df.iterrows():

    # Use new IDs to distinguish this generation method
    request_id = f"R{101 + i:04d}"

    for characteristic in request["SELECTED_CHARACTERISTICS"]:

        # Get valid values for the selected characteristic
        available_values = values[
            values["CHARACTERISTIC"] == characteristic
        ]

        if available_values.empty:
            raise ValueError(
                f"No values found for {characteristic}"
            )

        # Select a value without referring to PRODUCT_MASTER
        selected = available_values.sample(
            n=1,
            random_state=rng.randint(0, 100000)
        ).iloc[0]

        requirement_type = rng.choices(
            ["MUST", "PREFERRED"],
            weights=[0.65, 0.35],
            k=1
        )[0]

        independent_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": selected["VALUE"],
            "VALUE_DEFINITION": selected["DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type
        })

independent_ground_truth = pd.DataFrame(independent_rows)

display(independent_ground_truth)

,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0101,WIDTH_CM,V003,70,EQUAL,MUST
1,R0101,OVEN_TYPE,V002,STEAM,EQUAL,MUST
2,R0102,PANEL_COLOR,V001,BLACK,EQUAL,MUST
3,R0102,PANEL_TYPE,V002,METAL,EQUAL,PREFERRED
4,R0102,ENERGY_CLASS,V001,A,EQUAL,PREFERRED
5,R0103,PACKAGING_TYPE,V002,BOX,EQUAL,MUST
6,R0103,OVEN_FUNCTIONS,V001,01-Top+Bottom | Bottom | Top,EQUAL,MUST
7,R0103,WIDTH_CM,V003,70,EQUAL,PREFERRED
8,R0103,OVEN_TYPE,V002,STEAM,EQUAL,MUST
9,R0103,KNOB_STRUCTURE,V001,ROTARY,EQUAL,MUST


In [24]:
# Step 16 - Combine ground truth from both generation methods

product_based = pilot_ground_truth.copy()
product_based["GENERATION_METHOD"] = "PRODUCT_ANCHORED"

independent = independent_ground_truth.copy()
independent["GENERATION_METHOD"] = "REQUIREMENT_FIRST"

request_ground_truth = pd.concat(
    [product_based, independent],
    ignore_index=True
)

# Check for duplicate requirement definitions
duplicates = request_ground_truth.duplicated(
    subset=["REQUEST_ID", "CHARACTERISTIC"]
).sum()

summary = (
    request_ground_truth
    .groupby("GENERATION_METHOD")
    .agg(
        N_REQUESTS=("REQUEST_ID", "nunique"),
        N_REQUIREMENTS=("CHARACTERISTIC", "count")
    )
    .reset_index()
)

display(summary)

print("Total requests:", request_ground_truth["REQUEST_ID"].nunique())
print("Total requirements:", len(request_ground_truth))
print("Duplicate characteristic assignments:", duplicates)

,GENERATION_METHOD,N_REQUESTS,N_REQUIREMENTS
0,PRODUCT_ANCHORED,5,19
1,REQUIREMENT_FIRST,5,19


Total requests: 10
Total requirements: 38
Duplicate characteristic assignments: 0


In [25]:
# Step 17 - Prepare customer language mapping

language_mapping = request_ground_truth[
    ["CHARACTERISTIC", "VALUE_DEFINITION"]
].drop_duplicates().copy()

language_mapping = language_mapping.sort_values(
    ["CHARACTERISTIC", "VALUE_DEFINITION"]
).reset_index(drop=True)

display(language_mapping)

,CHARACTERISTIC,VALUE_DEFINITION
0,ENERGY_CLASS,A
1,HANDLE_TYPE,H3
2,HANDLE_TYPE,H4
3,KNOB_COLOR,BLACK
4,KNOB_COLOR,SILVER
5,KNOB_COLOR,STAINLESS STEEL
6,KNOB_STRUCTURE,ROTARY
7,NUMBER_OF_FUNCTIONS,11
8,OVEN_FUNCTIONS,01-Top+Bottom | Bottom | Top
9,OVEN_FUNCTIONS,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...


In [26]:
# Step 18 - Build complete language mapping from master data

language_master = values[
    ["CHARACTERISTIC", "VALUE", "DEFINITION"]
].copy()

language_master = language_master.rename(
    columns={"DEFINITION": "VALUE_DEFINITION"}
)

language_master["CUSTOMER_LANGUAGE"] = ""
language_master["REVIEW_STATUS"] = "PENDING"

language_master = language_master.sort_values(
    ["CHARACTERISTIC", "VALUE"]
).reset_index(drop=True)

print("Total characteristic-value pairs:", len(language_master))
print(
    "Characteristics covered:",
    language_master["CHARACTERISTIC"].nunique()
)

display(language_master.head(15))

Total characteristic-value pairs: 90
Characteristics covered: 24


,CHARACTERISTIC,VALUE,VALUE_DEFINITION,CUSTOMER_LANGUAGE,REVIEW_STATUS
0,BAKING_TRAY,V001,0,,PENDING
1,BAKING_TRAY,V002,1,,PENDING
2,BAKING_TRAY,V003,2,,PENDING
3,CLEANING,V001,ETC,,PENDING
4,CLEANING,V002,CATALYTIC,,PENDING
5,CLEANING,V003,PYROLYSIS,,PENDING
6,CLEANING,V004,HYDROCLEAN,,PENDING
7,ENERGY_CLASS,V001,A,,PENDING
8,ENERGY_CLASS,V002,A+,,PENDING
9,ENERGY_CLASS,V003,A++,,PENDING


In [27]:
# Step 19 - Generate initial English customer language

def create_customer_language(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"]).strip()

    if characteristic == "WIDTH_CM":
        return f"{value} cm wide", "AUTO"

    elif characteristic == "OVEN_VOLUME_L":
        return f"{value}-liter oven capacity", "AUTO"

    elif characteristic == "NUMBER_OF_FUNCTIONS":
        return f"{value} cooking functions", "AUTO"

    elif characteristic == "PANEL_COLOR":
        return f"{value.lower()} control panel", "AUTO"

    elif characteristic == "PANEL_TYPE":
        return f"{value.lower()} control panel", "AUTO"

    elif characteristic == "OVEN_TYPE":
        return f"{value.lower()} oven", "AUTO"

    elif characteristic == "ENERGY_CLASS":
        return f"energy class {value}", "AUTO"

    elif characteristic == "CLEANING":
        cleaning_terms = {
            "ETC": "easy-to-clean enamel",
            "CATALYTIC": "catalytic cleaning",
            "PYROLYSIS": "pyrolytic cleaning",
            "HYDROCLEAN": "hydroclean cleaning"
        }
        if value in cleaning_terms:
            return cleaning_terms[value], "AUTO"
        return "", "REVIEW"

    elif characteristic == "WIFI":
        return (
            "Wi-Fi connectivity" if value == "YES"
            else "no Wi-Fi connectivity"
        ), "AUTO"

    elif characteristic in [
        "BAKING_TRAY",
        "ROASTING_TRAY",
        "WIRE_SHELF",
        "TELESCOPIC_RAILS"
    ]:
        names = {
            "BAKING_TRAY": "baking trays",
            "ROASTING_TRAY": "roasting trays",
            "WIRE_SHELF": "wire shelves",
            "TELESCOPIC_RAILS": "telescopic rail sets"
        }
        return f"{value} {names[characteristic]}", "AUTO"

    # Technical or complex values require manual review
    return "", "REVIEW"


language_master[
    ["CUSTOMER_LANGUAGE", "REVIEW_STATUS"]
] = language_master.apply(
    lambda row: pd.Series(create_customer_language(row)),
    axis=1
)

display(
    language_master[
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "CUSTOMER_LANGUAGE",
            "REVIEW_STATUS"
        ]
    ]
)

print("\nReview status:")
print(language_master["REVIEW_STATUS"].value_counts())

,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE,REVIEW_STATUS
0,BAKING_TRAY,0,0 baking trays,AUTO
1,BAKING_TRAY,1,1 baking trays,AUTO
2,BAKING_TRAY,2,2 baking trays,AUTO
3,CLEANING,ETC,easy-to-clean enamel,AUTO
4,CLEANING,CATALYTIC,catalytic cleaning,AUTO
...,...,...,...,...
85,WIFI,YES,Wi-Fi connectivity,AUTO
86,WIFI,NO,no Wi-Fi connectivity,AUTO
87,WIRE_SHELF,0,0 wire shelves,AUTO
88,WIRE_SHELF,1,1 wire shelves,AUTO



Review status:
REVIEW_STATUS
REVIEW    46
AUTO      44
Name: count, dtype: int64


In [28]:
# Step 20 - Review characteristics requiring language definitions

review_summary = (
    language_master[
        language_master["REVIEW_STATUS"] == "REVIEW"
    ]
    .groupby("CHARACTERISTIC")
    .agg(
        N_VALUES=("VALUE", "count"),
        EXAMPLE_VALUES=(
            "VALUE_DEFINITION",
            lambda x: ", ".join(x.astype(str).head(3))
        )
    )
    .reset_index()
    .sort_values("N_VALUES", ascending=False)
)

display(review_summary)


,CHARACTERISTIC,N_VALUES,EXAMPLE_VALUES
6,OVEN_FUNCTIONS,12,"01-Top+Bottom | Bottom | Top, 02-Defrost | Top..."
1,HANDLE_TYPE,5,"H1, H2, H3"
4,KNOB_TYPE,5,"K1, K2, K3"
9,POWER_CABLE,5,"W/O CABLE, W/O PLUG, EU TYPE PLUG"
0,HANDLE_COLOR,4,"BLACK, WHITE, SILVER"
2,KNOB_COLOR,4,"BLACK, WHITE, SILVER"
5,OVEN_CONTROL,3,"W/O TIMER , DIGITAL TIMER, ELECTRONIC TOUCH TIMER"
3,KNOB_STRUCTURE,2,"ROTARY, RETRACTABLE"
7,PACKAGING_TYPE,2,"SHRINK, BOX"
8,PIZZA_TRAY,2,"YES, NO"


In [29]:
# Step 21 - Extend customer language mapping

def extend_customer_language(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"]).strip()
    status = row["REVIEW_STATUS"]

    if status != "REVIEW":
        return row["CUSTOMER_LANGUAGE"], status

    if characteristic == "HANDLE_COLOR":
        return f"{value.lower()} oven handle", "AUTO"

    elif characteristic == "KNOB_COLOR":
        return f"{value.lower()} control knobs", "AUTO"

    elif characteristic == "KNOB_STRUCTURE":
        descriptions = {
            "ROTARY": "rotary control knobs",
            "RETRACTABLE": "retractable control knobs"
        }
        return descriptions.get(value, ""), (
            "AUTO" if value in descriptions else "REVIEW"
        )

    elif characteristic == "OVEN_CONTROL":
        descriptions = {
            "W/O TIMER": "no timer",
            "DIGITAL TIMER": "digital timer",
            "ELECTRONIC TOUCH TIMER": "electronic touch timer"
        }
        return descriptions.get(value, ""), (
            "AUTO" if value in descriptions else "REVIEW"
        )

    elif characteristic == "PACKAGING_TYPE":
        return f"{value.lower()} packaging", "AUTO"

    elif characteristic == "PIZZA_TRAY":
        return (
            "pizza tray included" if value == "YES"
            else "no pizza tray included"
        ), "AUTO"

    elif characteristic == "TURNSPIT":
        return (
            "rotisserie turnspit included" if value == "YES"
            else "no rotisserie turnspit"
        ), "AUTO"

    elif characteristic == "POWER_CABLE":
        return value.lower(), "REVIEW"

    return row["CUSTOMER_LANGUAGE"], status


language_master[
    ["CUSTOMER_LANGUAGE", "REVIEW_STATUS"]
] = language_master.apply(
    lambda row: pd.Series(extend_customer_language(row)),
    axis=1
)

print(language_master["REVIEW_STATUS"].value_counts())

display(
    language_master[
        language_master["REVIEW_STATUS"] == "REVIEW"
    ][
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "CUSTOMER_LANGUAGE"
        ]
    ]
)

REVIEW_STATUS
AUTO      63
REVIEW    27
Name: count, dtype: int64


,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE
14,HANDLE_TYPE,H1,
15,HANDLE_TYPE,H2,
16,HANDLE_TYPE,H3,
17,HANDLE_TYPE,H4,
18,HANDLE_TYPE,H5,
25,KNOB_TYPE,K1,
26,KNOB_TYPE,K2,
27,KNOB_TYPE,K3,
28,KNOB_TYPE,K4,
29,KNOB_TYPE,K5,


In [31]:
# Step 22 - Add provisional language for technical design codes

def map_design_code(row):
    characteristic = row["CHARACTERISTIC"]
    value = str(row["VALUE_DEFINITION"]).strip()

    if characteristic == "HANDLE_TYPE":
        return f"{value} handle design", "PROVISIONAL"

    if characteristic == "KNOB_TYPE":
        return f"{value} control knob design", "PROVISIONAL"

    return row["CUSTOMER_LANGUAGE"], row["REVIEW_STATUS"]


mask = language_master["CHARACTERISTIC"].isin(
    ["HANDLE_TYPE", "KNOB_TYPE"]
)

language_master.loc[
    mask, ["CUSTOMER_LANGUAGE", "REVIEW_STATUS"]
] = language_master.loc[mask].apply(
    lambda row: pd.Series(map_design_code(row)),
    axis=1
).to_numpy()

display(
    language_master.loc[
        mask,
        [
            "CHARACTERISTIC",
            "VALUE_DEFINITION",
            "CUSTOMER_LANGUAGE",
            "REVIEW_STATUS"
        ]
    ]
)

print("\nReview status:")
print(language_master["REVIEW_STATUS"].value_counts())


,CHARACTERISTIC,VALUE_DEFINITION,CUSTOMER_LANGUAGE,REVIEW_STATUS
14,HANDLE_TYPE,H1,H1 handle design,PROVISIONAL
15,HANDLE_TYPE,H2,H2 handle design,PROVISIONAL
16,HANDLE_TYPE,H3,H3 handle design,PROVISIONAL
17,HANDLE_TYPE,H4,H4 handle design,PROVISIONAL
18,HANDLE_TYPE,H5,H5 handle design,PROVISIONAL
25,KNOB_TYPE,K1,K1 control knob design,PROVISIONAL
26,KNOB_TYPE,K2,K2 control knob design,PROVISIONAL
27,KNOB_TYPE,K3,K3 control knob design,PROVISIONAL
28,KNOB_TYPE,K4,K4 control knob design,PROVISIONAL
29,KNOB_TYPE,K5,K5 control knob design,PROVISIONAL



Review status:
REVIEW_STATUS
AUTO           63
REVIEW         17
PROVISIONAL    10
Name: count, dtype: int64


In [32]:
# Step 23 - Inspect remaining technical values

remaining_review = language_master.loc[
    language_master["REVIEW_STATUS"] == "REVIEW",
    [
        "CHARACTERISTIC",
        "VALUE",
        "VALUE_DEFINITION",
        "CUSTOMER_LANGUAGE"
    ]
].copy()

pd.set_option("display.max_colwidth", 200)

display(remaining_review.to_string(index=False))

'CHARACTERISTIC VALUE                                                                                                                                         VALUE_DEFINITION CUSTOMER_LANGUAGE\nOVEN_FUNCTIONS  V001                                                                                                                             01-Top+Bottom | Bottom | Top                  \nOVEN_FUNCTIONS  V002                                                                                            02-Defrost | Top+Bottom | Top+Bottom+Fan | Grill |  Grill+Fan                  \nOVEN_FUNCTIONS  V003                                                                                               03-Bottom | Top | Top+Bottom | Top+Bottom+Fan | Bottom+Fan                  \nOVEN_FUNCTIONS  V004                   04-OvenLight | Defrost | Top+Bottom | Top+Bottom+Fan | Bottom+Fan | Grill+Turnspit | Double Grill+Turnspit | Double Grill+Fan+Turnspit                  \nOVEN_FUNCTIONS  V005              

In [33]:
# Step 24 - Define the scope of the first NLP dataset

excluded_characteristics = ["OVEN_FUNCTIONS"]

generation_profile = characteristic_profile.copy()

generation_profile.loc[
    generation_profile["CHARACTERISTIC"].isin(excluded_characteristics),
    "ENABLED"
] = False

# Complete provisional language for power cable values
power_mask = language_master["CHARACTERISTIC"] == "POWER_CABLE"

language_master.loc[
    power_mask, "CUSTOMER_LANGUAGE"
] = (
    language_master.loc[power_mask, "VALUE_DEFINITION"]
    .astype(str)
    .str.strip()
)

language_master.loc[
    power_mask, "REVIEW_STATUS"
] = "PROVISIONAL"

print("Enabled characteristics:",
      generation_profile["ENABLED"].sum())

print("\nLanguage mapping status:")
print(language_master["REVIEW_STATUS"].value_counts())

print("\nExcluded from request generation:")
display(
    generation_profile.loc[
        ~generation_profile["ENABLED"],
        ["CHARACTERISTIC", "ENABLED"]
    ]
)

Enabled characteristics: 23

Language mapping status:
REVIEW_STATUS
AUTO           63
PROVISIONAL    15
REVIEW         12
Name: count, dtype: int64

Excluded from request generation:


,CHARACTERISTIC,ENABLED
9,OVEN_FUNCTIONS,False


In [34]:
# Step 25 - Select characteristics for the first NLP dataset

import random

rng = random.Random(789)

def select_request_characteristics(profile, rng, min_count=2, max_count=6):

    eligible = profile.loc[
        profile["ENABLED"] == True
    ].copy()

    n = rng.randint(min_count, min(max_count, len(eligible)))

    selected = rng.choices(
        eligible["CHARACTERISTIC"].tolist(),
        weights=eligible["REQUEST_WEIGHT"].tolist(),
        k=n
    )

    # Remove duplicates while preserving order
    selected = list(dict.fromkeys(selected))

    return selected


pilot_profiles = []

for i in range(1, 11):

    selected = select_request_characteristics(
        generation_profile,
        rng
    )

    pilot_profiles.append({
        "REQUEST_ID": f"R{i:04d}",
        "N_CHARACTERISTICS": len(selected),
        "SELECTED_CHARACTERISTICS": selected
    })

pilot_profiles_df = pd.DataFrame(pilot_profiles)

display(pilot_profiles_df)

print(
    "Contains OVEN_FUNCTIONS:",
    pilot_profiles_df["SELECTED_CHARACTERISTICS"].apply(
        lambda x: "OVEN_FUNCTIONS" in x
    ).any()
)

,REQUEST_ID,N_CHARACTERISTICS,SELECTED_CHARACTERISTICS
0,R0001,4,"[HANDLE_COLOR, WIDTH_CM, PANEL_COLOR, ROASTING_TRAY]"
1,R0002,4,"[PIZZA_TRAY, KNOB_STRUCTURE, HANDLE_COLOR, OVEN_TYPE]"
2,R0003,3,"[TURNSPIT, TELESCOPIC_RAILS, NUMBER_OF_FUNCTIONS]"
3,R0004,5,"[PANEL_COLOR, PIZZA_TRAY, KNOB_COLOR, HANDLE_COLOR, WIDTH_CM]"
4,R0005,5,"[CLEANING, PIZZA_TRAY, WIDTH_CM, NUMBER_OF_FUNCTIONS, WIFI]"
5,R0006,3,"[PANEL_COLOR, OVEN_TYPE, KNOB_STRUCTURE]"
6,R0007,5,"[CLEANING, PIZZA_TRAY, KNOB_TYPE, ENERGY_CLASS, WIDTH_CM]"
7,R0008,3,"[WIDTH_CM, OVEN_VOLUME_L, OVEN_TYPE]"
8,R0009,4,"[ROASTING_TRAY, PANEL_COLOR, CLEANING, PANEL_TYPE]"
9,R0010,4,"[CLEANING, PANEL_COLOR, KNOB_COLOR, ROASTING_TRAY]"


Contains OVEN_FUNCTIONS: False


In [35]:
# Step 26 - Generate structured ground truth for pilot requests

import random

rng = random.Random(890)

ground_truth_rows = []

for _, request in pilot_profiles_df.iterrows():

    request_id = request["REQUEST_ID"]

    for characteristic in request["SELECTED_CHARACTERISTICS"]:

        available_values = language_master.loc[
            (language_master["CHARACTERISTIC"] == characteristic)
            & (language_master["REVIEW_STATUS"].isin(["AUTO", "PROVISIONAL"]))
        ]

        if available_values.empty:
            raise ValueError(
                f"No usable language mapping for {characteristic}"
            )

        selected = available_values.iloc[
            rng.randrange(len(available_values))
        ]

        requirement_type = rng.choices(
            ["MUST", "PREFERRED"],
            weights=[0.70, 0.30],
            k=1
        )[0]

        ground_truth_rows.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": selected["VALUE"],
            "VALUE_DEFINITION": selected["VALUE_DEFINITION"],
            "OPERATOR": "EQUAL",
            "REQUIREMENT_TYPE": requirement_type,
            "CUSTOMER_LANGUAGE": selected["CUSTOMER_LANGUAGE"]
        })

nlp_pilot_ground_truth = pd.DataFrame(ground_truth_rows)

display(nlp_pilot_ground_truth.head(15))

print("Requests:", nlp_pilot_ground_truth["REQUEST_ID"].nunique())
print("Requirements:", len(nlp_pilot_ground_truth))
print(
    "Missing customer language:",
    nlp_pilot_ground_truth["CUSTOMER_LANGUAGE"].isna().sum()
    + nlp_pilot_ground_truth["CUSTOMER_LANGUAGE"].eq("").sum()
)


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE,CUSTOMER_LANGUAGE
0,R0001,HANDLE_COLOR,V001,BLACK,EQUAL,PREFERRED,black oven handle
1,R0001,WIDTH_CM,V003,70,EQUAL,PREFERRED,70 cm wide
2,R0001,PANEL_COLOR,V005,DARK SILVER,EQUAL,PREFERRED,dark silver control panel
3,R0001,ROASTING_TRAY,V002,1,EQUAL,MUST,1 roasting trays
4,R0002,PIZZA_TRAY,V002,NO,EQUAL,MUST,no pizza tray included
5,R0002,KNOB_STRUCTURE,V002,RETRACTABLE,EQUAL,MUST,retractable control knobs
6,R0002,HANDLE_COLOR,V002,WHITE,EQUAL,PREFERRED,white oven handle
7,R0002,OVEN_TYPE,V003,COMBINATION,EQUAL,MUST,combination oven
8,R0003,TURNSPIT,V002,NO,EQUAL,MUST,no rotisserie turnspit
9,R0003,TELESCOPIC_RAILS,V002,1,EQUAL,MUST,1 telescopic rail sets


Requests: 10
Requirements: 40
Missing customer language: 0


In [36]:
# Step 27 - Generate customer request texts from ground truth

def compose_customer_request(requirements):

    must_sentences = []
    preferred_sentences = []

    for _, row in requirements.iterrows():

        phrase = str(row["CUSTOMER_LANGUAGE"]).strip()
        requirement_type = row["REQUIREMENT_TYPE"]

        if requirement_type == "MUST":
            must_sentences.append(
                f"It must have {phrase}."
            )

        elif requirement_type == "PREFERRED":
            preferred_sentences.append(
                f"I would prefer {phrase}."
            )

        else:
            raise ValueError(
                f"Unsupported requirement type: {requirement_type}"
            )

    return " ".join(
        ["I'm looking for an oven."]
        + must_sentences
        + preferred_sentences
    )


customer_request_rows = []

for request_id, group in nlp_pilot_ground_truth.groupby(
    "REQUEST_ID", sort=True
):

    request_text = compose_customer_request(group)

    customer_request_rows.append({
        "REQUEST_ID": request_id,
        "REQUEST_TEXT": request_text
    })


nlp_pilot_customer_requests = pd.DataFrame(
    customer_request_rows
)

pd.set_option("display.max_colwidth", 500)

display(nlp_pilot_customer_requests)

print(
    "Generated customer requests:",
    len(nlp_pilot_customer_requests)
)

,REQUEST_ID,REQUEST_TEXT
0,R0001,I'm looking for an oven. It must have 1 roasting trays. I would prefer black oven handle. I would prefer 70 cm wide. I would prefer dark silver control panel.
1,R0002,I'm looking for an oven. It must have no pizza tray included. It must have retractable control knobs. It must have combination oven. I would prefer white oven handle.
2,R0003,I'm looking for an oven. It must have no rotisserie turnspit. It must have 1 telescopic rail sets. I would prefer 8 cooking functions.
3,R0004,I'm looking for an oven. It must have stainless steel control panel. It must have no pizza tray included. It must have black control knobs. It must have stainless steel oven handle. It must have 60 cm wide.
4,R0005,I'm looking for an oven. It must have hydroclean cleaning. It must have 10 cooking functions. It must have Wi-Fi connectivity. I would prefer pizza tray included. I would prefer 70 cm wide.
5,R0006,I'm looking for an oven. It must have silver control panel. It must have steam oven. It must have rotary control knobs.
6,R0007,I'm looking for an oven. It must have hydroclean cleaning. It must have no pizza tray included. It must have K5 control knob design. It must have 70 cm wide. I would prefer energy class A+.
7,R0008,I'm looking for an oven. It must have 45 cm wide. It must have 65-liter oven capacity. I would prefer convection oven.
8,R0009,I'm looking for an oven. It must have 2 roasting trays. It must have black control panel. It must have glass control panel. I would prefer hydroclean cleaning.
9,R0010,I'm looking for an oven. It must have dark silver control panel. It must have black control knobs. It must have 1 roasting trays. I would prefer hydroclean cleaning.


Generated customer requests: 10
